In [1]:
!pip install mediapipe opencv-python

In [2]:
!pip install --upgrade tensorflow

In [3]:
!pip install mediapipe opencv-python

In [1]:
import cv2
import numpy as np
import os
#from matplotlib import pyplot as plt
import time
import mediapipe as mp

In [2]:
mp_holistic = mp.solutions.holistic # Holistic model
mp_drawing = mp.solutions.drawing_utils # Drawing utilities

In [3]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable 
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
    return image, results

In [4]:
def draw_styled_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) # Draw pose connections


In [5]:
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    return pose

In [6]:
# Path for exported data, numpy arrays
DATA_PATH = os.path.join('MP_Data') 

# Actions that we try to detect
actions = np.array(['boxing', 'walk-stand', 'felldown'])

# Thirty videos worth of data
no_sequences = 30

# Videos are going to be 60 frames in length
sequence_length = 60

# Folder start
start_folder = 30

In [7]:
for action in actions: 
    for sequence in range(no_sequences):
        try: 
            os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
        except:
            pass

# cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    
    # NEW LOOP
    # Loop through actions
    for action in actions:
        # Loop through sequences aka videos
        for sequence in range(no_sequences):
            # Loop through video length aka sequence length
            for frame_num in range(sequence_length):

                # Read feed
                ret, frame = cap.read()

                # Make detections
                image, results = mediapipe_detection(frame, holistic)
#                 print(results)

                # Draw landmarks
                draw_styled_landmarks(image, results)
                
                # NEW Apply wait logic
                if frame_num == 0: 
                    cv2.putText(image, 'STARTING COLLECTION', (120,200), 
                               cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255, 0), 4, cv2.LINE_AA)
                    cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15,12), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    # Show to screen
                    cv2.imshow('OpenCV Feed', image)
                    cv2.waitKey(6000)
                else: 
                    cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15,12), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    # Show to screen
                    cv2.imshow('OpenCV Feed', image)
                
                # NEW Export keypoints
                keypoints = extract_keypoints(results)
                npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
                np.save(npy_path, keypoints)

                # Break gracefully
                if cv2.waitKey(10) & 0xFF == ord('q'):
                    break
                    
    cap.release()
    cv2.destroyAllWindows()

In [7]:
from sklearn.model_selection import train_test_split
#from keras.utils import to_categorical
# from keras.utils import np_utils
#import keras.utils.np_utils.to_categorical
from tensorflow.keras.utils import to_categorical

In [8]:
label_map = {label:num for num, label in enumerate(actions)}

In [9]:
print(label_map)

{'boxing': 0, 'walk-stand': 1, 'felldown': 2}


In [10]:
sequences, labels = [], []
for action in actions:
    for sequence in np.array(os.listdir(os.path.join(DATA_PATH, action))).astype(int):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), "{}.npy".format(frame_num)))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

In [32]:
np.array(labels).shape

(90,)

In [14]:
np.array(sequences).shape

(90, 60, 132)

In [24]:
X = np.array(sequences)

In [25]:
y = to_categorical(labels).astype(int)

In [109]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.03)

In [110]:
X_train.shape

(87, 60, 132)

In [111]:
y_train.shape

(87, 3)

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import TensorBoard

In [113]:
log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)

In [114]:
actions.shape[0]

3

In [12]:
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(60,132)))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(actions.shape[0], activation='softmax'))

In [116]:
model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])

In [117]:
model.fit(X_train, y_train, epochs=500, callbacks=[tb_callback])

Epoch 1/500
3/3 [==============================] - 2s 52ms/step - loss: 1.0937 - categorical_accuracy: 0.4138
Epoch 2/500
3/3 [==============================] - 0s 49ms/step - loss: 1.0050 - categorical_accuracy: 0.6897
Epoch 3/500
3/3 [==============================] - 0s 50ms/step - loss: 0.9273 - categorical_accuracy: 0.6322
Epoch 4/500
3/3 [==============================] - 0s 52ms/step - loss: 0.8061 - categorical_accuracy: 0.5977
Epoch 5/500
3/3 [==============================] - 0s 49ms/step - loss: 0.9838 - categorical_accuracy: 0.5747
Epoch 6/500
3/3 [==============================] - 0s 50ms/step - loss: 1.0614 - categorical_accuracy: 0.3793
Epoch 7/500
3/3 [==============================] - 0s 50ms/step - loss: 1.0037 - categorical_accuracy: 0.5517
Epoch 8/500
3/3 [==============================] - 0s 49ms/step - loss: 0.8679 - categorical_accuracy: 0.5632
Epoch 9/500
3/3 [==============================] - 0s 51ms/step - loss: 0.8297 - categorical_accuracy: 0.4253
Epoch 10/5

3/3 [==============================] - 0s 71ms/step - loss: 0.4707 - categorical_accuracy: 0.7356
Epoch 148/500
3/3 [==============================] - 0s 76ms/step - loss: 0.8596 - categorical_accuracy: 0.6092
Epoch 149/500
3/3 [==============================] - 0s 70ms/step - loss: 1.2485 - categorical_accuracy: 0.4023
Epoch 150/500
3/3 [==============================] - 0s 72ms/step - loss: 114.8015 - categorical_accuracy: 0.3793
Epoch 151/500
3/3 [==============================] - 0s 70ms/step - loss: 1.2670 - categorical_accuracy: 0.3333
Epoch 152/500
3/3 [==============================] - 0s 68ms/step - loss: 1.1922 - categorical_accuracy: 0.3333
Epoch 153/500
3/3 [==============================] - 0s 67ms/step - loss: 1.0575 - categorical_accuracy: 0.3793
Epoch 154/500
3/3 [==============================] - 0s 68ms/step - loss: 1.5703 - categorical_accuracy: 0.4368
Epoch 155/500
3/3 [==============================] - 0s 68ms/step - loss: 1.1121 - categorical_accuracy: 0.3908
Epoc

3/3 [==============================] - 0s 69ms/step - loss: 0.5908 - categorical_accuracy: 0.6322
Epoch 294/500
3/3 [==============================] - 0s 67ms/step - loss: 0.5559 - categorical_accuracy: 0.7471
Epoch 295/500
3/3 [==============================] - 0s 70ms/step - loss: 0.5696 - categorical_accuracy: 0.8276
Epoch 296/500
3/3 [==============================] - 0s 68ms/step - loss: 0.5495 - categorical_accuracy: 0.8851
Epoch 297/500
3/3 [==============================] - 0s 69ms/step - loss: 0.5027 - categorical_accuracy: 0.8966
Epoch 298/500
3/3 [==============================] - 0s 68ms/step - loss: 0.4452 - categorical_accuracy: 0.8966
Epoch 299/500
3/3 [==============================] - 0s 68ms/step - loss: 0.4242 - categorical_accuracy: 0.8621
Epoch 300/500
3/3 [==============================] - 0s 70ms/step - loss: 0.3878 - categorical_accuracy: 0.9195
Epoch 301/500
3/3 [==============================] - 0s 69ms/step - loss: 0.3558 - categorical_accuracy: 0.9425
Epoch 

3/3 [==============================] - 0s 74ms/step - loss: 0.0759 - categorical_accuracy: 0.9770
Epoch 440/500
3/3 [==============================] - 0s 72ms/step - loss: 0.0782 - categorical_accuracy: 0.9770
Epoch 441/500
3/3 [==============================] - 0s 68ms/step - loss: 0.0740 - categorical_accuracy: 0.9770
Epoch 442/500
3/3 [==============================] - 0s 69ms/step - loss: 0.0598 - categorical_accuracy: 0.9770
Epoch 443/500
3/3 [==============================] - 0s 68ms/step - loss: 0.0634 - categorical_accuracy: 0.9655
Epoch 444/500
3/3 [==============================] - 0s 68ms/step - loss: 0.0555 - categorical_accuracy: 0.9885
Epoch 445/500
3/3 [==============================] - 0s 68ms/step - loss: 0.0587 - categorical_accuracy: 0.9770
Epoch 446/500
3/3 [==============================] - 0s 67ms/step - loss: 0.0588 - categorical_accuracy: 0.9770
Epoch 447/500
3/3 [==============================] - 0s 68ms/step - loss: 0.0469 - categorical_accuracy: 0.9885
Epoch 

In [118]:
model.save('action.h5')   #model.load_weights('action.h5')

In [18]:
model.load_weights('action.h5')

In [18]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 60, 64)            50432     
                                                                 
 lstm_1 (LSTM)               (None, 60, 128)           98816     
                                                                 
 lstm_2 (LSTM)               (None, 64)                49408     
                                                                 
 dense (Dense)               (None, 64)                4160      
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 3)                 99        
                                                                 
Total params: 204,995
Trainable params: 204,995
Non-trai

In [119]:
X_test.shape

(3, 60, 132)

In [120]:
res = model.predict(X_test)

In [121]:
res.shape

(3, 3)

In [122]:
res

array([[9.9914110e-01, 6.5532111e-04, 2.0349535e-04],
       [3.2737688e-03, 9.9671268e-01, 1.3544338e-05],
       [1.9786311e-08, 2.9539716e-08, 1.0000000e+00]], dtype=float32)

In [125]:
actions[np.argmax(res[0])]

'boxing'

In [126]:
y_test

array([[1, 0, 0],
       [0, 1, 0],
       [0, 0, 1]])

In [127]:
actions[np.argmax(y_test[0])]

'boxing'

In [128]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

In [138]:
X_test.shape

(3, 60, 132)

In [150]:
yhat = model.predict(X_test)

In [151]:
yhat

array([[9.9914110e-01, 6.5532111e-04, 2.0349535e-04],
       [3.2737688e-03, 9.9671268e-01, 1.3544338e-05],
       [1.9786311e-08, 2.9539716e-08, 1.0000000e+00]], dtype=float32)

In [152]:
s=np.array(yhat)
s.shape

(3, 3)

In [153]:
ytrue = np.argmax(y_test, axis=1).tolist()
yhat = np.argmax(yhat, axis=1).tolist()

In [154]:
ytrue

[0, 1, 2]

In [155]:
yhat

[0, 1, 2]

In [156]:
multilabel_confusion_matrix(ytrue, yhat)

array([[[2, 0],
        [0, 1]],

       [[2, 0],
        [0, 1]],

       [[2, 0],
        [0, 1]]], dtype=int64)

In [157]:
accuracy_score(ytrue, yhat)

1.0

In [19]:
import pywhatkit
from datetime import datetime
def alert():
    now = datetime.now() 
    pywhatkit.sendwhatmsg("+917382072519","Ur Kid fell down",int(now.strftime("%H")), int(now.strftime("%M"))+1)

In [20]:
colors = [(245,117,16), (117,245,16), (16,117,245)]
def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()
    for num, prob in enumerate(res):
        cv2.rectangle(output_frame, (0,60+num*40), (int(prob*100), 90+num*40), colors[num], -1)
        cv2.putText(output_frame, actions[num], (0, 85+num*40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)
        
    return output_frame

In [23]:
# 1. New detection variables
sequence = []
sentence = []
threshold = 1
count=0
cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()

        # Make detections
        image, results = mediapipe_detection(frame, holistic)
        print(results)
        
        # Draw landmarks
        draw_styled_landmarks(image, results)
        
        # 2. Prediction logic
        keypoints = extract_keypoints(results)
#         sequence.insert(0,keypoints)
#         sequence = sequence[:30]
        sequence.append(keypoints)
        sequence = sequence[-60:]
        
        if len(sequence) == 60:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            
            
        #3. Viz logic
#             if res[np.argmax(res)] > threshold: 
#                 if len(sentence) > 0: 
#                     if actions[np.argmax(res)] != sentence[-1]:
#                         sentence.append(actions[np.argmax(res)])
#                 else:
#                     sentence.append(actions[np.argmax(res)])

#             if len(sentence) > 5: 
#                 sentence = sentence[-5:]

            # Viz probabilities
            image = prob_viz(res, actions, image, colors)
            
#         cv2.rectangle(image, (0,0), (640, 40), (245, 117, 16), -1)
#        # cv2.putText(image, ' '.join(sentence), (3,30), 
#                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
            if(actions[np.argmax(res)]=='felldown'):
                count+=1
        if(count > 2):
                alert()
                break
        # Show to screen
        cv2.imshow('OpenCV Feed', image)

        # Break gracefully
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.soluti

CallTimeException: Call Time must be Greater than Wait Time as WhatsApp Web takes some Time to Load!